In [ ]:
print(df.shape) 
print(df.columns)
print(df.dtypes)

In [ ]:
# Sort by video_id, then id
df_s = df.sort(["video_id", "id"], 
               descending=[False, False])
#Add new Index
df_s = df_s.with_columns(pl.arange(0,df_s.height).alias("new_index"))
# Move new_Index to the first column
df_s = df_s.select(["new_index"] + [c for c in df_s.columns if c != "new_index"])

In [ ]:
df_s.head(1000).to_pandas()

In [ ]:
import polars as pl
import numpy as np

def missing_frames_summary_polars(df: pl.DataFrame, video_col="video_id", frame_col="frame"):
    """
    Compute missing frames statistics per video with mean, std, max, and % missing.
    Returns per-video stats sorted by number of missing frames (descending).
    """

    # Step 1: Count unique frames per video
    unique_frame_counts = (
        df.group_by(video_col)
          .agg(pl.col(frame_col).n_unique().alias("num_unique_frames"))
    )

    # Step 2: Compute min/max frame per video
    frame_range = (
        df.group_by(video_col)
          .agg([
              pl.col(frame_col).min().alias("min_frame"),
              pl.col(frame_col).max().alias("max_frame")
          ])
    )

    # Step 3: Join counts with ranges
    frame_stats = frame_range.join(unique_frame_counts, on=video_col)

    # Step 4: Compute missing frames
    frame_stats = frame_stats.with_columns([
        (pl.col("max_frame") - pl.col("min_frame") + 1 - pl.col("num_unique_frames"))
        .alias("num_missing_frames")
    ])

    # Step 5: Compute missing frames as percentage
    frame_stats = frame_stats.with_columns([
        ((pl.col("num_missing_frames") / (pl.col("max_frame") - pl.col("min_frame") + 1)) * 100)
        .alias("pct_missing")
    ])

    # Step 6: Sort by number of missing frames descending
    frame_stats = frame_stats.sort("num_missing_frames",descending=True)

    # Step 7: Convert to NumPy arrays for summary statistics
    num_missing_array = frame_stats["num_missing_frames"].to_numpy()
    pct_missing_array = frame_stats["pct_missing"].to_numpy()

    # Step 8: Compute summary
    summary = {
        "num_missing_frames_mean": np.mean(num_missing_array),
        "num_missing_frames_std": np.std(num_missing_array),
        "num_missing_frames_max": np.max(num_missing_array),
        "pct_missing_mean": np.mean(pct_missing_array),
    }

    return summary, frame_stats



In [ ]:
summary, frame_stats = missing_frames_summary_polars(df_s, video_col="video_id", frame_col="frame")

print("=== Missing Frames Summary ===")
for k, v in summary.items():
    print(f"{k}: {v:.2f}")

# The per-video stats are now sorted by number of missing frames descending
print(frame_stats)


=== Missing Frames Summary ===
num_missing_frames_mean: 0.40
num_missing_frames_std: 2.00
num_missing_frames_max: 57.00
pct_missing_mean: 0.19